In [1]:
!pip install torch torchvision opencv-python tqdm matplotlib scikit-learn seaborn

Defaulting to user installation because normal site-packages is not writeable


In [3]:
import os
import random
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from torch.optim.lr_scheduler import ReduceLROnPlateau


In [7]:
# -------------------------------
# REPRODUCIBILITY
# -------------------------------
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

In [11]:
# -------------------------------
# CONFIG
# -------------------------------
DATASET_DIR = r"Downloads/CASIA-dataset"  # update to your dataset folder

attack_ratio = 0.5
BATCH_SIZE = 16
EPOCHS = 25
LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_PATH = "resnet50_best.pth"
NUM_WORKERS = 0   # safe for Windows + Jupyter
EARLY_STOPPING_PATIENCE = 6
GRAD_CLIP_NORM = 3.0

print("Using device:", DEVICE)

Using device: cpu


In [15]:
# ------------------------------
# DATASET CLASS (FIXED & ROBUST)
# ------------------------------

class FingerprintDataset(Dataset):

    def __init__(self, samples, transform=None):
        """
        samples: list of tuples (img_path, label)
        label: 0 -> Real, 1 -> Reconstructed/Attack
        """
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def _apply_simulated_attack(self, image):
        """
        Simulate realistic attack artifacts on a grayscale image
        """

        # Blur
        blur_val = int(np.random.choice([3, 5, 7]))
        sigma = float(np.random.uniform(0.3, 2.5))
        attacked = cv2.GaussianBlur(image, (blur_val, blur_val), sigmaX=sigma)

        # Contrast / Brightness
        alpha = float(np.random.uniform(0.6, 1.3))
        beta = int(np.random.randint(-25, 25))
        attacked = attacked.astype(np.float32) * alpha + beta

        # Noise sometimes
        if np.random.rand() > 0.35:
            noise_sigma = float(np.random.uniform(3.0, 18.0))
            noise = np.random.normal(0, noise_sigma, attacked.shape).astype(np.float32)
            attacked = attacked + noise

        # Occasional JPEG artifacts
        if np.random.rand() > 0.6:
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), np.random.randint(30, 90)]
            _, encimg = cv2.imencode('.jpg', attacked.astype(np.uint8), encode_param)
            attacked = cv2.imdecode(encimg, 0)

        attacked = np.clip(attacked, 0, 255).astype(np.uint8)

        return attacked

    def __getitem__(self, idx):

        img_path, label = self.samples[idx]

        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

        if label == 1:
            image = self._apply_simulated_attack(image)

        if self.transform:
            image = self.transform(image)

        return image, label